# Generic streaming batch statistics

`cx.tl.batch_process` applies a mergeable statistic within experimental batches without loading the complete cell-by-gene matrix. This example computes a batch-corrected standard deviation for every perturbation and gene.

In [ ]:
from pathlib import Path
import sys

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

ROOT = Path('../..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
import crispyx as cx

OUTPUT_DIR = ROOT / 'docs' / 'notebooks' / 'tutorial_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Create a batched example

The batches deliberately have different means and variances. Computing variability within each batch prevents shifts in batch means from inflating the result.

In [ ]:
rng = np.random.default_rng(7)
rows, perturbations, batches = [], [], []
for batch_index, batch in enumerate(['lane_1', 'lane_2', 'lane_3']):
    for group_index, perturbation in enumerate(['control', 'KO_A', 'KO_B']):
        n_cells = 20 + 4 * batch_index + 2 * group_index
        block = rng.normal(
            loc=4 * batch_index + group_index,
            scale=0.5 + batch_index + 0.25 * group_index,
            size=(n_cells, 6),
        )
        rows.append(block)
        perturbations.extend([perturbation] * n_cells)
        batches.extend([batch] * n_cells)

X = np.vstack(rows)
obs = pd.DataFrame(
    {'perturbation': perturbations, 'batch': batches},
    index=[f'cell_{i}' for i in range(X.shape[0])],
)
var = pd.DataFrame(index=[f'gene_{i}' for i in range(X.shape[1])])
input_path = OUTPUT_DIR / 'batch_statistics_input.h5ad'
ad.AnnData(sp.csr_matrix(X), obs=obs, var=var).write(input_path)

## Define a streaming standard-deviation reducer

A reducer has three callbacks in per-group mode. `initialize` creates state for a gene chunk, `update` merges each cell chunk using the parallel-variance formula, and `finalize` returns the within-batch statistic plus its aggregation weight. Here the weight is the number of cells in that perturbation/batch.

In [ ]:
def initialize_std(n_genes):
    return {'n': 0, 'mean': np.zeros(n_genes), 'm2': np.zeros(n_genes)}

def update_std(state, block):
    block_n = block.shape[0]
    block_mean = block.mean(axis=0)
    block_m2 = np.square(block - block_mean).sum(axis=0)
    if state['n'] == 0:
        state.update(n=block_n, mean=block_mean, m2=block_m2)
        return
    total_n = state['n'] + block_n
    delta = block_mean - state['mean']
    state['m2'] += block_m2 + delta**2 * state['n'] * block_n / total_n
    state['mean'] += delta * block_n / total_n
    state['n'] = total_n

def finalize_std(state):
    std = np.sqrt(state['m2'] / (state['n'] - 1))
    return cx.BatchStatistic(std, weight=state['n'])

std_reducer = cx.BatchReducer(
    initialize=initialize_std,
    update=update_std,
    finalize=finalize_std,
)

In [ ]:
std_result = cx.tl.batch_process(
    input_path,
    std_reducer,
    groupby='perturbation',  # DE-compatible alias
    batch_column='batch',
    mode='group',
    statistic_name='std',
    chunk_size=3,       # genes per chunk
    cell_chunk_size=16, # cells per reducer update
    output_path=OUTPUT_DIR / 'batch_corrected_std.h5ad',
    force=True,
)

corrected_std = pd.DataFrame(
    std_result.backed.X[:],
    index=std_result.backed.obs_names,
    columns=std_result.backed.var_names,
)
corrected_std

For group $g$ and gene $j$, the result is $\sum_b n_{gb}s_{gbj}/\sum_b n_{gb}$: a cell-count-weighted average of within-batch sample standard deviations. The between-batch mean shift is therefore excluded.

In [ ]:
reference = []
for perturbation in corrected_std.index:
    batch_stds, batch_counts = [], []
    for batch in pd.unique(obs['batch']):
        mask = (obs['perturbation'] == perturbation) & (obs['batch'] == batch)
        batch_stds.append(X[mask].std(axis=0, ddof=1))
        batch_counts.append(mask.sum())
    reference.append(np.average(batch_stds, axis=0, weights=batch_counts))

np.testing.assert_allclose(corrected_std.to_numpy(), reference)
print('Streaming and direct calculations agree.')

## Comparison mode

For a group-versus-reference statistic, add a `compare(group_state, reference_state)` callback and call with `mode='comparison'`. The arguments `reference`/`control_label`, `groupby`/`perturbation_column`, and `perturbations` behave like crispyx DE functions. If no reference is supplied, crispyx infers the control label using the same rules as DE. Only batches containing both the group and reference contribute.